# Visual Emotion Recognition — Transfer Learning with ResNet50
**المكون الثاني في Poetry Vision — ريناد**

الهدف: نموذج يقترح **الشعور اللي توحي فيه الصورة** (مو مشاعر المستخدم). نطبق نفس خطوات Part 3:
1. **Feature Extraction:** نجمد ResNet50 وندرب رأس جديد.
2. **Fine-tuning:** نفك تجميد آخر الطبقات وندرب بـ learning rate أقل.

قبل ما تبدين: **Runtime ← Change runtime type ← T4 GPU**

In [ ]:
import os, io, json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

In [ ]:
# The 8 emotions labelled in EmoSet (same order everywhere)
CLASSES = ['amusement', 'awe', 'contentment', 'excitement', 'anger', 'disgust', 'fear', 'sadness']
ARABIC  = {'amusement': 'المرح', 'awe': 'الرهبة', 'contentment': 'السكينة', 'excitement': 'الحماس',
           'anger': 'الغضب', 'disgust': 'الاشمئزاز', 'fear': 'الخوف', 'sadness': 'الحزن'}

DATA_DIR = '/content/emoset'          # emoset/train/<emotion>/*.jpg , emoset/val/... , emoset/test/...
OUT      = '/content/emotion_keras'   # model, results and charts
os.makedirs(f'{OUT}/results', exist_ok=True)

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

## 1) Load the data

البيانات: **EmoSet-118K**. كل صورة عليها واحد من 8 مشاعر. ناخذ عينة متوازنة من كل تقسيم رسمي:
**1000 صورة لكل شعور للتدريب، و200 للـ validation، و500 للـ test.**

اختاري **طريقة وحدة** من التالي:
- **الطريقة A (الأسرع):** ترفعين ملف `emoset.zip` اللي نزّلتيه من الجلسة السابقة.
- **الطريقة B:** تحمّلين من Hugging Face من جديد. هذي تاخذ وقت طويل (المرة الأولى أخذت تقريباً ساعتين).

In [ ]:
# Option A: upload emoset.zip (prepared earlier)
from google.colab import files
uploaded = files.upload()
!unzip -q -o emoset.zip -d /content
!ls /content/emoset

In [ ]:
# Option B: download a balanced sample from Hugging Face (slow) — run ONLY if you don't have emoset.zip
# !pip install -q datasets
# from datasets import load_dataset
#
# def download_emoset(split, per_class):
#     ds = load_dataset('Woleek/EmoSet-118K', split=split, streaming=True).shuffle(seed=42, buffer_size=2000)
#     counts = {c: 0 for c in CLASSES}
#     for row in ds:
#         emotion = row['emotion']
#         if counts[emotion] < per_class:
#             folder = f'{DATA_DIR}/{split}/{emotion}'
#             os.makedirs(folder, exist_ok=True)
#             img = row['image'].convert('RGB')
#             img.thumbnail((256, 256))
#             img.save(f'{folder}/{row["image_id"]}.jpg', quality=90)
#             counts[emotion] += 1
#         if all(n >= per_class for n in counts.values()):
#             break
#     print(split, counts)
#
# download_emoset('train', 1000)
# download_emoset('val', 200)
# download_emoset('test', 500)

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/train', class_names=CLASSES, image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, shuffle=True, seed=42)

val_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/val', class_names=CLASSES, image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, shuffle=False)

test_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/test', class_names=CLASSES, image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, shuffle=False)   # never used for training or choosing settings

test_files = test_ds.file_paths

In [ ]:
# Show some training images with their emotion
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(f"Label: {CLASSES[labels[i]]}")
        plt.axis('off')
plt.savefig(f'{OUT}/results/sample_images.png', dpi=120)
plt.show()

In [ ]:
# Images per emotion in each split
for name, folder in [('train', 'train'), ('val', 'val'), ('test', 'test')]:
    counts = {c: len(os.listdir(f'{DATA_DIR}/{folder}/{c}')) for c in CLASSES}
    print(name, counts)

## 2) Preprocessing and augmentation

- ResNet50 تدرّب على ImageNet بطريقة تجهيز خاصة فيه، فنستخدم `preprocess_input` بدل القسمة على 255.
- **Augmentation:** قلب أفقي وتكبير بسيط بس. **ما نغيّر الألوان ولا السطوع**، لأنها جزء من الشعور: لو غيّرناها، ممكن الصورة الحزينة تصير مبهجة ويصير الـ label غلط.

In [ ]:
preprocess = tf.keras.applications.resnet50.preprocess_input
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(lambda x, y: (preprocess(x), y)).prefetch(AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (preprocess(x), y)).prefetch(AUTOTUNE)
test_ds  = test_ds.map(lambda x, y: (preprocess(x), y)).prefetch(AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomZoom(0.1),
])

y_test = np.concatenate([y.numpy() for _, y in test_ds])

## 3) Feature Extraction (freeze the backbone)

In [ ]:
# Load Pretrained ResNet50
base_model = tf.keras.applications.ResNet50(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False  # Freeze backbone

In [ ]:
# Build Classification Head
model = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    data_augmentation,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(len(CLASSES), activation='softmax')
])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

### Before training
نقيس النموذج قبل أي تدريب. الرأس عشوائي، فالمتوقع قريب من التخمين (1 من 8 = 0.125). هذا خط الأساس اللي نقارن فيه.

In [ ]:
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay

results = {}

def evaluate_stage(name):
    probs = model.predict(test_ds, verbose=0)
    y_pred = probs.argmax(axis=1)
    acc = float((y_pred == y_test).mean())
    macro_f1 = float(f1_score(y_test, y_pred, average='macro'))
    results[name] = {'test_accuracy': acc, 'test_macro_f1': macro_f1}
    print(f"{name}: Test Accuracy = {acc:.4f} | Macro F1 = {macro_f1:.4f}")
    return probs

probs_before = evaluate_stage('before_training')

In [ ]:
history = model.fit(
    train_ds,
    epochs=5,
    validation_data=val_ds
)

In [ ]:
probs_feature_extraction = evaluate_stage('feature_extraction')

## 4) Fine-tuning (unfreeze top layers only)

- نفك تجميد **آخر 30 طبقة** بس من ResNet50. فك التجميد الكامل مع بيانات قليلة يسبب overfitting.
- طبقات **BatchNormalization** نخليها مجمدة، لأن تحديثها بدفعات صغيرة يخرب اللي تعلمه النموذج.
- **learning rate أقل** (1e-5). أقل من الـ lab (1e-4)، لأن ResNet50 أكبر من MobileNetV2.

In [ ]:
# Unfreeze top layers only
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers[-30:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

In [ ]:
# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Continue Training
fine_tune_history = model.fit(
    train_ds,
    epochs=5,
    validation_data=val_ds
)

In [ ]:
probs_fine_tuning = evaluate_stage('fine_tuning')

test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc:.4f}")

## 5) Results

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'] + fine_tune_history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'] + fine_tune_history.history['val_accuracy'], label='Val Accuracy')
plt.axvline(len(history.history['accuracy']) - 0.5, color='gray', linestyle=':', label='Start Fine Tuning')
plt.legend(); plt.xlabel('Epoch'); plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'] + fine_tune_history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'] + fine_tune_history.history['val_loss'], label='Val Loss')
plt.axvline(len(history.history['loss']) - 0.5, color='gray', linestyle=':')
plt.legend(); plt.xlabel('Epoch'); plt.ylabel('Loss')

plt.savefig(f'{OUT}/results/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Before vs after training
names = list(results.keys())
plt.figure(figsize=(7, 4))
plt.bar(names, [results[n]['test_macro_f1'] for n in names], label='Macro F1')
plt.plot(names, [results[n]['test_accuracy'] for n in names], 'o', color='orange', label='Accuracy')
plt.axhline(1 / len(CLASSES), linestyle=':', color='gray', label='Chance (1/8)')
for i, n in enumerate(names):
    plt.text(i, results[n]['test_macro_f1'] + 0.01, f"{results[n]['test_macro_f1']:.3f}", ha='center')
plt.ylabel('Test score'); plt.legend()
plt.savefig(f'{OUT}/results/before_vs_after.png', dpi=150)
plt.show()

In [ ]:
y_pred = probs_fine_tuning.argmax(axis=1)
report = classification_report(y_test, y_pred, target_names=CLASSES, digits=3)
print(report)
open(f'{OUT}/results/classification_report.txt', 'w').write(report)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=CLASSES,
                                        normalize='true', values_format='.2f',
                                        cmap='Blues', ax=ax, xticks_rotation=45)
plt.title('Confusion Matrix (normalized) — Fine-tuned ResNet50')
plt.tight_layout()
plt.savefig(f'{OUT}/results/confusion_matrix.png', dpi=150)
plt.show()

## 6) Images with more than one emotion

**margin** = الفرق بين أعلى احتمالين. إذا كان صغير، معناه النموذج متردد بين شعورين، والصورة فعلاً تحتمل أكثر من قراءة. في هالحالة التطبيق يعرض أكثر من اقتراح.

In [ ]:
sorted_probs = np.sort(probs_fine_tuning, axis=1)
margin = sorted_probs[:, -1] - sorted_probs[:, -2]
ambiguous = margin < 0.15
top2 = np.argsort(probs_fine_tuning, axis=1)[:, -2:]

results['ambiguity'] = {
    'fraction_ambiguous': float(ambiguous.mean()),
    'accuracy_clear': float((y_pred[~ambiguous] == y_test[~ambiguous]).mean()),
    'accuracy_ambiguous': float((y_pred[ambiguous] == y_test[ambiguous]).mean()) if ambiguous.any() else None,
    'top2_accuracy_all': float(np.mean([y_test[i] in top2[i] for i in range(len(y_test))])),
}
print(results['ambiguity'])

plt.figure(figsize=(14, 7))
for k, i in enumerate(np.argsort(margin)[:8]):
    a, b = np.argsort(probs_fine_tuning[i])[::-1][:2]
    plt.subplot(2, 4, k + 1)
    plt.imshow(plt.imread(test_files[i]))
    plt.title(f"true: {CLASSES[y_test[i]]}\n{CLASSES[a]} {probs_fine_tuning[i, a]:.2f} | {CLASSES[b]} {probs_fine_tuning[i, b]:.2f}", fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT}/results/ambiguous_examples.png', dpi=130)
plt.show()

## 7) Colour and light by emotion

هل الصور الحزينة أعتم؟ وهل صور المرح ألوانها أكثر؟ نحسب السطوع والتشبع لكل صورة في الـ test ونقارن بين المشاعر.

In [ ]:
from PIL import Image

brightness, saturation = [], []
for f in test_files:
    hsv = np.asarray(Image.open(f).convert('RGB').resize((64, 64)).convert('HSV'), dtype='float32') / 255.0
    saturation.append(hsv[..., 1].mean())
    brightness.append(hsv[..., 2].mean())
brightness, saturation = np.array(brightness), np.array(saturation)

plt.figure(figsize=(14, 4))
for k, (values, title) in enumerate([(brightness, 'Brightness'), (saturation, 'Saturation')]):
    plt.subplot(1, 2, k + 1)
    plt.boxplot([values[y_test == c] for c in range(len(CLASSES))], showfliers=False)
    plt.xticks(range(1, len(CLASSES) + 1), CLASSES, rotation=45)
    plt.title(f'{title} by emotion (test set)')
plt.tight_layout()
plt.savefig(f'{OUT}/results/colour_light_by_emotion.png', dpi=150)
plt.show()

results['mean_brightness'] = {c: float(brightness[y_test == i].mean()) for i, c in enumerate(CLASSES)}
results['mean_saturation'] = {c: float(saturation[y_test == i].mean()) for i, c in enumerate(CLASSES)}

## 8) Save the model and the results

In [ ]:
model.save(f'{OUT}/emotion_resnet50.keras')
json.dump({'classes': CLASSES, 'arabic': ARABIC, 'img_size': list(IMG_SIZE),
           'model_version': 'emoset8-resnet50-keras-v1'},
          open(f'{OUT}/model_config.json', 'w'), ensure_ascii=False, indent=2)
json.dump(results, open(f'{OUT}/results/metrics.json', 'w'), indent=2)
print(json.dumps(results, indent=2))

In [ ]:
# Download results (charts + metrics + model)
from google.colab import files
!cd /content && rm -f emotion_keras_results.zip && zip -qr emotion_keras_results.zip emotion_keras
files.download('/content/emotion_keras_results.zip')

## 9) analyze_emotions(image_bytes)

هذي الدالة اللي يناديها التطبيق. تعطيها صورة، وترجع **اقتراحات** للمستخدم (يقدر يأكدها أو يغيرها)، مع درجة ثقة النموذج، وهل الصورة غامضة.

In [ ]:
def analyze_emotions(image_bytes, top_k=3):
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    x = np.asarray(img.resize(IMG_SIZE), dtype='float32')[None]
    probs = model.predict(preprocess(x), verbose=0)[0]

    order = np.argsort(probs)[::-1]
    suggestions, total = [], 0.0
    for i in order[:top_k]:
        if suggestions and (total >= 0.8 or probs[i] < 0.10):
            break
        suggestions.append({'label': CLASSES[i], 'label_ar': ARABIC[CLASSES[i]],
                            'score': round(float(probs[i]), 4), 'source': 'trained'})
        total += float(probs[i])

    hsv = np.asarray(img.resize((64, 64)).convert('HSV'), dtype='float32') / 255.0
    margin = float(probs[order[0]] - probs[order[1]])
    return {
        'emotion_suggestions': suggestions,
        'visual_affective_attributes': {'brightness': round(float(hsv[..., 2].mean()), 3),
                                        'saturation': round(float(hsv[..., 1].mean()), 3)},
        'uncertainty': {'top1_prob': round(float(probs[order[0]]), 4),
                        'margin_top1_top2': round(margin, 4),
                        'is_ambiguous': margin < 0.15},
        'model_version': 'emoset8-resnet50-keras-v1',
    }

In [ ]:
# Try it on your own photo
uploaded = files.upload()
for name, data in uploaded.items():
    plt.imshow(Image.open(io.BytesIO(data))); plt.axis('off'); plt.show()
    print(json.dumps(analyze_emotions(data), ensure_ascii=False, indent=2))